In [ ]:
import numpy as np
import scipy as sp
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.legend_handler import HandlerTuple
import mpmath as mp
from tqdm.auto import tqdm
import cblind as cb

from matplotlib.transforms import Bbox, TransformedBbox, blended_transform_factory
from mpl_toolkits.axes_grid1.inset_locator import (BboxConnector, BboxConnectorPatch,
                                                   BboxPatch)
#%matplotlib widget

In [ ]:
nb = 1e-3
ns = 1e-6
N = 200
etamax = .5
etavals = np.linspace(0,etamax,N)
eta1,eta2 = np.meshgrid(etavals,etavals)

etamin2 = 1e-2
etamax2 = 5e-2
etavals2 = np.linspace(etamin2,etamax2,N)
eta12,eta22 = np.meshgrid(etavals2,etavals2)

# This code is all here to facilitate checkerboard colorings
gridodd = np.array([[ (np.floor(i*10/N)+np.floor(j*10/N)) % 2 for i in range(N)] for j in range(N)])
bpatch = mpatches.Patch(color="black")
wpatch = mpatches.Patch(facecolor="white", edgecolor = "black")
checkpatch = (bpatch,wpatch)
hmap = {tuple: HandlerTuple(ndivide=None,pad=0)}
checkcolors = [[ "black" if val == 0 else "white" for val in vals] for vals in gridodd]

In [ ]:
def connect_bbox(bbox1, bbox2,
                 loc1a, loc2a, loc1b, loc2b,
                 prop_lines, prop_patches=None):
    if prop_patches is None:
        prop_patches = {
            **prop_lines,
            "alpha": prop_lines.get("alpha", 1) * 0.2,
            "clip_on": False,
            "ec": "black",
        }

    c1 = BboxConnector(
        bbox1, bbox2, loc1=loc1a, loc2=loc2a, clip_on=False, **prop_lines)
    c2 = BboxConnector(
        bbox1, bbox2, loc1=loc1b, loc2=loc2b, clip_on=False, **prop_lines)

    bbox_patch1 = BboxPatch(bbox1, **prop_patches)
    bbox_patch2 = BboxPatch(bbox2, **prop_patches)

    p = BboxConnectorPatch(bbox1, bbox2,
                           loc1a=loc1a, loc2a=loc2a, loc1b=loc1b, loc2b=loc2b,
                           clip_on=False,
                           **prop_patches)

    return c1, c2, bbox_patch1, bbox_patch2, p
def zoom_effect(ax1, ax2, xmin,ymin,xmax,ymax,**kwargs):
    """
    ax1 : the main Axes
    ax2 : the zoomed Axes

    """


    bbox = Bbox.from_extents(xmin, ymin, xmax, ymax)
    mybbox1 = TransformedBbox(bbox, ax1.transData)
    mybbox2 = ax2.bbox

    prop_patches = {**kwargs, "ec": "black", "alpha": 0.1}

    c1, c2, bbox_patch1, bbox_patch2, p = connect_bbox(
        mybbox1, mybbox2,
        loc1a=1, loc2a=1, loc1b=3, loc2b=3,
        prop_lines=kwargs, prop_patches=prop_patches)

    ax1.add_patch(bbox_patch1)
    ax2.add_patch(bbox_patch2)
    ax2.add_patch(c1)
    ax2.add_patch(c2)
    #ax2.add_patch(p)

    return c1, c2, bbox_patch1, bbox_patch2, p

In [ ]:
@np.vectorize(cache=True)
def Qrel(eta1,eta2,nb,ns):
# This freaks out with very small values...
# Could be a situation where I should redo this with mpmath to avoid such extreme values
# but it could also be worth reworking this to use logs instead of arctanh, which may simplify things considerably
# mpmath doesn't seem to fix the issue, which is somewhat surprising.
    eta1 = mp.mpf(eta1)
    eta2 = mp.mpf(eta2)
    nb = mp.mpf(nb)
    ns = mp.mpf(ns)
    def nu0(eta):
        to_sq = (1-eta)*ns +1+nb
        to_sqrt = to_sq**2 + 4*eta*nb*ns
        return mp.sqrt(to_sqrt)
    def nu1(eta):
        diff = nb - (1-eta) *ns
        return diff + nu0(eta)
    def nu2(eta):
        diff = (1-eta) *ns - nb
        return diff + nu0(eta)
    def lamsq(eta):
        to_sqrt = eta*ns*(1+ns)
        num = 1+nb+ns+eta*ns + 2*mp.sqrt(to_sqrt)
        return num/nu0(eta)

    # Define values that will be used
    nu11 = nu1(eta1)
    nu12 = nu1(eta2)
    nu21 = nu2(eta1)
    nu22 = nu2(eta2)

    
    logterm = mp.log(1+nb-eta2) - mp.log(1+nb-eta1)
    constterm = 2*(nu11 *mp.acoth(nu11) + nu21 * mp.acoth(nu21))
    sumterm = (nu11 + nu21) *(mp.acoth(nu12) + mp.acoth(nu22))/2
    lamterm = lamsq(eta1)/lamsq(eta2) + lamsq(eta2)/lamsq(eta1)
    diffterm = (nu11 - nu21) *(mp.acoth(nu12) - mp.acoth(nu22))
    nonlogterm = sumterm*lamterm + diffterm-constterm
    return (logterm+nonlogterm)/2

In [ ]:
qrels = Qrel(eta1,eta2,nb,ns)
qrels2 = Qrel(eta12,eta22,nb,ns)

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(projection='3d')
ax.plot_surface(eta1,eta2,qrels)
ax.set_xlabel('Transmisivity 1')
ax.set_ylabel('Transmisivity 2')
ax.set_zlabel('Quantum Relative Entropy')
ax.set_title("TMSV Quantum Limit")
plt.show()

In [ ]:
def rel_ent_pnr(eta1,eta2,nb,ns):
    def P0I(eta,nb,ns):
        return (1+ns)*(1+nb)
    def P0II(eta,nb,ns):
        return 1+ eta*ns+nb
    def P0SI(eta,nb,ns):
        return 1+ns
    def P0(eta,nb,ns):
        return 1/P0I(eta,nb,ns)
    def PI(eta,nb,ns):
        return 1/P0II(eta,nb,ns)- P0(eta,nb,ns)
    def PS(eta,nb,ns):
        return 1/P0SI(eta,nb,ns)- P0(eta,nb,ns)
    def PC(eta,nb,ns):
        return 1-PS(eta,nb,ns)-PI(eta,nb,ns)-P0(eta,nb,ns)
    # For our calculation here we can ignore the PS and P0 terms since they have no eta dependance
    PIval = PI(eta1,nb,ns) * (np.log(PI(eta1,nb,ns)) - np.log(PI(eta2,nb,ns)))
    PCval = PC(eta1,nb,ns) * (np.log(PC(eta1,nb,ns)) - np.log(PC(eta2,nb,ns)))
    return PIval +PCval

In [ ]:
pnr_rels = rel_ent_pnr(eta1,eta2,nb,ns)
pnr_rels2 = rel_ent_pnr(eta12,eta22,nb,ns)

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(projection='3d')
ax.plot_surface(eta1,eta2,pnr_rels)
ax.set_xlabel('Transmisivity 1')
ax.set_ylabel('Transmisivity 2')
ax.set_zlabel('Relative Entropy')
ax.set_title("Click No-Click Detection")
plt.show()

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(projection='3d')
pnr_surf = ax.plot_surface(eta1,eta2,pnr_rels, label="Click No-Click")
ql_surf = ax.plot_surface(eta1,eta2,qrels, label="Quantum Limit",facecolors = checkcolors)

ax.set_xlabel('Transmisivity 1')
ax.set_ylabel('Transmisivity 2')
ax.set_zlabel('Relative Entropy')
ax.set_title("Click No-Click Detection vs Quantum Limit")
leg = ax.legend([pnr_surf,checkpatch], ["Click No-Click", "Quantum Limit"], handler_map = hmap)
plt.show()

In [ ]:
@np.vectorize
def rel_ent_su11(eta1,eta2,nb,ns):
    eta1 = mp.mpf(eta1)
    eta2 = mp.mpf(eta2)
    nb = mp.mpf(nb)
    ns = mp.mpf(ns)
    def P0I(eta):
        return nb*(1+2*ns) + (1+(1+mp.sqrt(eta))*ns)**2
    def P0II(eta):
        return 1+ nb*(1+ns)+ ns*(1+ns)*(1+mp.sqrt(eta))**2
    def P0SI(eta):
        return 1+ns*(2+nb+ns+2*(1+ns)*mp.sqrt(eta)+ns*eta)
    def P0(eta):
        return 1/P0I(eta)
    def PI(eta):
        return 1/P0II(eta)- P0(eta)
    def PS(eta):
        return 1/P0SI(eta)- P0(eta)
    def PC(eta):
        return 1-PS(eta)-PI(eta)-P0(eta)
    
    P0val = P0(eta1) * (mp.log(P0(eta1)) - mp.log(P0(eta2)))
    PSval = PS(eta1) * (mp.log(PS(eta1)) - mp.log(PS(eta2)))
    PIval = PI(eta1) * (mp.log(PI(eta1)) - mp.log(PI(eta2)))
    PCval = PC(eta1) * (mp.log(PC(eta1)) - mp.log(PC(eta2)))
    return PIval +PCval+PSval+P0val

In [ ]:
su11_rels = rel_ent_su11(eta1,eta2,nb,ns)
su11_rels2 = rel_ent_su11(eta12,eta22,nb,ns)

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(projection='3d')
ax.plot_surface(eta1,eta2,su11_rels)
ax.set_xlabel('Transmisivity 1')
ax.set_ylabel('Transmisivity 2')
ax.set_title("SU(1,1) Detection")
ax.set_zlabel('Relative Entropy')
plt.show()

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(projection='3d')
su11_surf = ax.plot_surface(eta1,eta2,su11_rels, label = "SU(1,1)")
pnr_surf = ax.plot_surface(eta1,eta2,pnr_rels, label = "Click No-Click", facecolors=checkcolors)
ax.set_xlabel('Transmisivity 1')
ax.set_ylabel('Transmisivity 2')
ax.set_zlabel('Relative Entropy')
ax.set_title("SU(1,1) vs Click No-Click")
#ax.legend(handles = [su11_surf,pnr_surf])
leg = ax.legend([su11_surf,checkpatch], ["SU(1,1)", "Click No-Click"], handler_map = hmap)

plt.show()

In [ ]:

fig = plt.figure()
ax = fig.add_subplot(projection='3d')
pnr_surf = ax.plot_surface(eta1,eta2,su11_rels, label="SU(1,1)")
ql_surf = ax.plot_surface(eta1,eta2,qrels, label="Quantum Limit", facecolors=checkcolors)
ax.set_xlabel('Transmisivity 1')
ax.set_ylabel('Transmisivity 2')
ax.set_zlabel('Relative Entropy')
ax.set_title("SU(1,1) vs Quantum Limit")
ax.legend(handles = [pnr_surf,ql_surf])
leg = ax.legend([pnr_surf,checkpatch], ["SU(1,1)", "Quantum Limit"], handler_map = hmap)

plt.show()

In [ ]:
y = np.array([[qrels[i,j]/su11_rels[i,j] if i!=j else np.nan for i in range(N)]for j in range(N)],dtype=float)
#avval = np.nanmedian(y)
#for i in range(N):
#    y[i,i] = avval
# We hand-roll a median filter here in order to use nanmedian and avoid issues with nans
y2 = np.zeros([N-2,N-2])
k=2 #filter range
drop=0
for i in range(1,N-1):
    ik = min(k,N-i,i)
    il = i-ik
    iu = i+ik+1
    for j in range(1,N-1):
        jk = min(k,N-j,j)
        jl = j-jk
        ju = j+jk+1
        subrange = y[il:iu,jl:ju]
        y2[i-1,j-1] = np.nanmedian(subrange)
y2 = y2[drop:N-2-drop,drop:N-2-drop]
#y2 = sp.signal.medfilt2d(y,kernel_size=7)
#y2 = sp.signal.wiener(y2)
    
to_plot = y2
fig = plt.figure()
ax = fig.add_subplot()
CS = ax.pcolormesh(eta1[1+drop:N-1-drop,1+drop:N-1-drop],eta2[1+drop:N-1-drop,1+drop:N-1-drop],to_plot, cmap = "cb.iris")
#CS2 = ax.contour(eta1[1+drop:N-1-drop,1+drop:N-1-drop],eta2[1+drop:N-1-drop,1+drop:N-1-drop],to_plot, levels=[1], linestyles = 'dashed')

#pnr_surf = ax.plot_surface(eta1[1:N-1-drop,1:N-1-drop],eta2[1:N-1-drop,1:N-1-drop],to_plot, label="SU(1,1)")
ax.set_xlabel('Transmisivity 1')
ax.set_ylabel('Transmisivity 2')
#ax.set_zlabel(r"$\frac{D_Q}{D}$")
cbar = fig.colorbar(CS)
cbar.ax.set_ylabel(r"$\frac{D_Q}{D}$")
ax.set_title("SU(1,1) vs Quantum Limit")

plt.show()

In [ ]:
y = np.array([[su11_rels[i,j]/pnr_rels[i,j] if i!=j else np.nan for i in range(N)]for j in range(N)],dtype=float)
y[0,0] = np.mean([y[0,1],y[1,0]])
y[N-1,N-1] = np.mean([y[N-1,N-2],y[N-2,N-1]])
s= 0
e = 0
for i in range(1,N-1):
    y[i,i] = np.mean([y[i,i-1],y[i,i+1],y[i-1,i],y[i+1,i]])
y2 = np.array([[su11_rels2[i,j]/pnr_rels2[i,j] if i!=j else np.nan for i in range(N)]for j in range(N)],dtype=float)
y2[0,0] = np.mean([y2[0,1],y2[1,0]])
y2[N-1,N-1] = np.mean([y2[N-1,N-2],y2[N-2,N-1]])
s2= 10
e2 = 0
for i in range(1,N-1):
    y2[i,i] = np.mean([y2[i,i-1],y2[i,i+1],y2[i-1,i],y2[i+1,i]])
    
to_plot = y
fig = plt.figure()
axs = fig.subplots(2)
#CS = ax.pcolormesh(eta1[1+drop:N-1-drop,1+drop:N-1-drop],eta2[1+drop:N-1-drop,1+drop:N-1-drop],to_plot, cmap = "cb.iris")
CS = axs[0].pcolormesh(eta1[s:N-e,s:N-e],eta2[s:N-e,s:N-e],to_plot[s:N-e,s:N-e], cmap = "cb.iris")
axs[0].contour(eta1[s:N-e,s:N-e],eta2[s:N-e,s:N-e],to_plot[s:N-e,s:N-e], levels=[1], linestyles = 'dashed')


#pnr_surf = ax.plot_surface(eta1[1:N-1-drop,1:N-1-drop],eta2[1:N-1-drop,1:N-1-drop],to_plot, label="SU(1,1)")
axs[0].set_xlabel('Transmisivity 1')
axs[0].set_ylabel('Transmisivity 2')
#ax.set_zlabel(r"$\frac{D_Q}{D}$")
cbar = fig.colorbar(CS,ax = axs[0])
cbar.ax.set_ylabel(r"$\frac{D_{\text{SU(1,1)}}}{D_{\text{Click No-Click}}}$")
axs[0].set_title("SU(1,1) vs Click No-Click")

to_plot2 = y2
CS2 = axs[1].pcolormesh(eta12,eta22,to_plot2, cmap = "cb.iris")
axs[1].contour(eta12,eta22,to_plot2, levels=[1], linestyles = 'dashed')


#pnr_surf = ax.plot_surface(eta1[1:N-1-drop,1:N-1-drop],eta2[1:N-1-drop,1:N-1-drop],to_plot, label="SU(1,1)")
axs[1].set_xlabel('Transmisivity 1')
axs[1].set_ylabel('Transmisivity 2')
#ax.set_zlabel(r"$\frac{D_Q}{D}$")
cbar = fig.colorbar(CS2,ax=axs[1])
cbar.ax.set_ylabel(r"$\frac{D_{\text{SU(1,1)}}}{D_{\text{Click No-Click}}}$")
zoom_effect(axs[0], axs[1], etamin2,etamin2,etamax2,etamax2)

plt.show()

In [ ]:
def rel_ent_coh_mp_single(eta1,eta2,nb,ns):
    A = 1/(1+nb)
    B = A/2
    C = 1/ns
    D = C/2
    def P0(eta,r):
        #return mp.exp(-r**2*eta/(2+2*nb))/(1+nb)
        return mp.exp(-r**2*eta*B)*A
    def rel_ent(r):
        P01 = P0(eta1,r)
        P02 = P0(eta2,r)
        PC1 = 1-P01
        PC2 = 1-P02
        # PCval is a bit of a lost cause, but P0val can certainly be calculated more efficiently
        # P0val = -r**2*P01 *(eta1-eta2)*B
        # Could potentially work, but it doesn't buy us much computationally
        # The real work seems to be the numeric integration which this doesn't let us eliminate
        # In order to do that we would need to calculate the second term explicitly as well
        # as the first (the first has a form that looks very analytically calculatable)
        P0val = P01*(mp.log(P01) - mp.log(P02))
        PCval = PC1*(mp.log(PC1)-mp.log(PC2))
        res = P0val +PCval
        #res[np.isnan(res)] = 0
        return res
    def to_int(r):
        #return mp.exp(-r**2/(2*ns)) *r/ns *rel_ent(r)
        return mp.exp(-r**2*D) *r*C *rel_ent(r)
    res = mp.quad(to_int,[0,mp.inf])
    return res
def rel_ent_coh_mp(eta1,eta2,nb,ns):
    shape = np.shape(eta1)
    to_ret = np.zeros(shape)
    for index in tqdm(np.ndindex(shape),total=np.prod(shape)):
        eta1v = eta1[index]
        eta2v = eta2[index]
        to_ret[index] = rel_ent_coh_mp_single(eta1v,eta2v,nb,ns)
    return to_ret

In [ ]:
# Use the first option to generate the data, and the second to load it from a file

#coh_rels = rel_ent_coh_mp(eta1,eta2,nb,ns)

#data = np.load("../_data/coh_ent.npz")
#coh_rels = data['rel_ent']

In [ ]:
rel_ent_coh_mp_single(.1,.2,nb,ns)

In [ ]:
#fig = plt.figure()
#ax = fig.add_subplot(projection='3d')
#ax.plot_surface(eta1,eta2,coh_rels)
#ax.set_xlabel('Transmisivity 1')
#ax.set_ylabel('Transmisivity 2')
#ax.set_zlabel('Relative Entropy')
#ax.set_title("Coherent Light")
#plt.show()

In [ ]:
# For saving data

#np.savez_compressed("../_data/coh_ent",eta1=eta1,eta2=eta2,nb=nb,ns=ns,rel_ent=coh_rels)

In [ ]:
#fig = plt.figure()
#ax = fig.add_subplot(projection='3d')
#su11_surf = ax.plot_surface(eta1,eta2,su11_rels, label = "SU(1,1)")
#coh_surf = ax.plot_surface(eta1,eta2,coh_rels, label = "Coherent")
#ax.set_xlabel('Transmisivity 1')
#ax.set_ylabel('Transmisivity 2')
#ax.set_zlabel('Relative Entropy')
#ax.set_title("SU(1,1) vs Coherent")
#ax.legend(handles = [su11_surf,coh_surf])
#plt.show()

In [ ]:
def Qrel_coh(eta1,eta2,nb,ns):
    return ns*(eta1-eta2)**2 *(np.log(1+nb) - np.log(nb))

In [ ]:
coh_qrels = Qrel_coh(eta1,eta2,nb,ns)
coh_qrels2 = Qrel_coh(eta12,eta22,nb,ns)

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(projection='3d')
ax.plot_surface(eta1,eta2,coh_qrels)
ax.set_xlabel('Transmisivity 1')
ax.set_ylabel('Transmisivity 2')
ax.set_zlabel('Quantum Relative Entropy')
ax.set_title("Coherent Quantum Limit")
plt.show()

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(projection='3d')
tmsv_surf = ax.plot_surface(eta1,eta2,qrels, label = "TMSV")
coh_surf = ax.plot_surface(eta1,eta2,coh_qrels, label = "Coherent")
ax.set_xlabel('Transmisivity 1')
ax.set_ylabel('Transmisivity 2')
ax.set_zlabel('Quantum Relative Entropy')
ax.set_title("TMSV vs Coherent, Quantum Limit")
ax.legend(handles = [tmsv_surf,coh_surf])
plt.show()

In [ ]:
y = np.array([[su11_rels[i,j]/coh_qrels[i,j] if i!=j else np.nan for i in range(N)]for j in range(N)],dtype=float)
#avval = np.nanmedian(y)
#for i in range(N):
#    y[i,i] = avval
# We hand-roll a median filter here in order to use nanmedian and avoid issues with nans
k=0 #filter range
drop=40
y[0,0] = np.mean([y[0,1],y[1,0]])
y[N-1,N-1] = np.mean([y[N-1,N-2],y[N-2,N-1]])
s= 0
e = 0
for i in range(1,N-1):
    y[i,i] = np.mean([y[i,i-1],y[i,i+1],y[i-1,i],y[i+1,i]])
#y2 = sp.signal.medfilt2d(y,kernel_size=7)
#y2 = sp.signal.wiener(y2)
    
to_plot = y
fig = plt.figure()
ax = fig.add_subplot()
#CS = ax.pcolormesh(eta1[1+drop:N-1-drop,1+drop:N-1-drop],eta2[1+drop:N-1-drop,1+drop:N-1-drop],to_plot, cmap = "cb.iris")
CS = ax.pcolormesh(eta1[s:N-e,s:N-e],eta2[s:N-e,s:N-e],to_plot[s:N-e,s:N-e], cmap = "cb.iris")
CS2 = ax.contour(eta1[s:N-e,s:N-e],eta2[s:N-e,s:N-e],to_plot[s:N-e,s:N-e], levels=[1], linestyles = 'dashed')


#pnr_surf = ax.plot_surface(eta1[1:N-1-drop,1:N-1-drop],eta2[1:N-1-drop,1:N-1-drop],to_plot, label="SU(1,1)")
ax.set_xlabel('Transmisivity 1')
ax.set_ylabel('Transmisivity 2')
#ax.set_zlabel(r"$\frac{D_Q}{D}$")
cbar = fig.colorbar(CS)
cbar.ax.set_ylabel(r"$\frac{D_{\text{SU(1,1)}}}{D_{\text{Coherent}}}$")
ax.set_title("SU(1,1) vs Coherent")

plt.show()

In [ ]:
y = np.array([[su11_rels[i,j]/coh_qrels[i,j] if i!=j else np.nan for i in range(N)]for j in range(N)],dtype=float)
y[0,0] = np.mean([y[0,1],y[1,0]])
y[N-1,N-1] = np.mean([y[N-1,N-2],y[N-2,N-1]])
s= 0
e = 0
for i in range(1,N-1):
    y[i,i] = np.mean([y[i,i-1],y[i,i+1],y[i-1,i],y[i+1,i]])

    
y2 = np.array([[su11_rels2[i,j]/coh_qrels2[i,j] if i!=j else np.nan for i in range(N)]for j in range(N)],dtype=float)
y2[0,0] = np.mean([y2[0,1],y2[1,0]])
y2[N-1,N-1] = np.mean([y2[N-1,N-2],y2[N-2,N-1]])
s2= 10
e2 = 0
for i in range(1,N-1):
    y2[i,i] = np.mean([y2[i,i-1],y2[i,i+1],y2[i-1,i],y2[i+1,i]])
    
to_plot = y
fig = plt.figure()
axs = fig.subplots(2)
#CS = ax.pcolormesh(eta1[1+drop:N-1-drop,1+drop:N-1-drop],eta2[1+drop:N-1-drop,1+drop:N-1-drop],to_plot, cmap = "cb.iris")
CS = axs[0].pcolormesh(eta1[s:N-e,s:N-e],eta2[s:N-e,s:N-e],to_plot[s:N-e,s:N-e], cmap = "cb.iris")
axs[0].contour(eta1[s:N-e,s:N-e],eta2[s:N-e,s:N-e],to_plot[s:N-e,s:N-e], levels=[1], linestyles = 'dashed')


#pnr_surf = ax.plot_surface(eta1[1:N-1-drop,1:N-1-drop],eta2[1:N-1-drop,1:N-1-drop],to_plot, label="SU(1,1)")
axs[0].set_xlabel('Transmisivity 1')
axs[0].set_ylabel('Transmisivity 2')
#ax.set_zlabel(r"$\frac{D_Q}{D}$")
cbar = fig.colorbar(CS,ax = axs[0])
cbar.ax.set_ylabel(r"$\frac{D_{\text{SU(1,1)}}}{D_{\text{Click No-Click}}}$")
axs[0].set_title("SU(1,1) vs Coherent")

to_plot2 = y2
CS2 = axs[1].pcolormesh(eta12,eta22,to_plot2, cmap = "cb.iris")
axs[1].contour(eta12,eta22,to_plot2, levels=[1], linestyles = 'dashed')


#pnr_surf = ax.plot_surface(eta1[1:N-1-drop,1:N-1-drop],eta2[1:N-1-drop,1:N-1-drop],to_plot, label="SU(1,1)")
axs[1].set_xlabel('Transmisivity 1')
axs[1].set_ylabel('Transmisivity 2')
#ax.set_zlabel(r"$\frac{D_Q}{D}$")
cbar = fig.colorbar(CS2,ax=axs[1])
cbar.ax.set_ylabel(r"$\frac{D_{\text{SU(1,1)}}}{D_{\text{Click No-Click}}}$")
zoom_effect(axs[0], axs[1], etamin2,etamin2,etamax2,etamax2)

plt.show()

In [ ]:
# fig = plt.figure()
# ax = fig.add_subplot(projection='3d')
# q_surf = ax.plot_surface(eta1,eta2,coh_qrels, label = "Quantum Limit")
# coh_surf = ax.plot_surface(eta1,eta2,coh_rels, label = "Click No-Click")
# ax.set_xlabel('Transmisivity 1')
# ax.set_ylabel('Transmisivity 2')
# ax.set_zlabel('Relative Entropy')
# ax.set_title("Coherent Light Click No-Click vs Quantum Limit")
# ax.legend(handles = [q_surf, coh_surf])
# plt.show()

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(projection='3d')
su11_surf = ax.plot_surface(eta1,eta2,su11_rels, label = "SU(1,1)")
coh_surf = ax.plot_surface(eta1,eta2,coh_qrels, label = "Coherent")
ax.set_xlabel('Transmisivity 1')
ax.set_ylabel('Transmisivity 2')
ax.set_zlabel('Relative Entropy')
ax.set_title("SU(1,1) vs Coherent Quantum Limit")
ax.legend(handles = [su11_surf,coh_surf])
plt.show()

In [ ]:
def rel_ent_homo(eta1, eta2, nb, ns):
    return 2*(eta1 - eta2)**2*ns *np.sqrt(np.pi/(1+nb))

In [ ]:
homo_rels = rel_ent_homo(eta1,eta2,nb,ns)

In [ ]:

fig = plt.figure()
ax = fig.add_subplot(projection='3d')
ax.plot_surface(eta1,eta2,homo_rels)
ax.set_xlabel('Transmisivity 1')
ax.set_ylabel('Transmisivity 2')
ax.set_zlabel('Relative Entropy')
ax.set_title("Coherent Light Homodyne")
plt.show()

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(projection='3d')
homo_surf = ax.plot_surface(eta1,eta2,homo_rels, label = "Homodyne")
coh_surf = ax.plot_surface(eta1,eta2,coh_qrels, label = "Coherent")
ax.set_xlabel('Transmisivity 1')
ax.set_ylabel('Transmisivity 2')
ax.set_zlabel('Relative Entropy')
ax.set_title("Coherent, Homodyne vs Quantum Limit")
ax.legend(handles = [homo_surf,coh_surf])
plt.show()

In [ ]:
y = np.array([[homo_rels[i,j]/coh_qrels[i,j] if i!=j else np.nan for i in range(N)]for j in range(N)],dtype=float)
#avval = np.nanmedian(y)
#for i in range(N):
#    y[i,i] = avval
# We hand-roll a median filter here in order to use nanmedian and avoid issues with nans
k=0 #filter range
y[0,0] = np.mean([y[0,1],y[1,0]])
y[N-1,N-1] = np.mean([y[N-1,N-2],y[N-2,N-1]])
s= 0
e = 0
for i in range(1,N-1):
    y[i,i] = np.mean([y[i,i-1],y[i,i+1],y[i-1,i],y[i+1,i]])
#y2 = sp.signal.medfilt2d(y,kernel_size=7)
#y2 = sp.signal.wiener(y2)
    
to_plot = y
fig = plt.figure()
ax = fig.add_subplot()
#CS = ax.pcolormesh(eta1[1+drop:N-1-drop,1+drop:N-1-drop],eta2[1+drop:N-1-drop,1+drop:N-1-drop],to_plot, cmap = "cb.iris")
CS = ax.pcolormesh(eta1[s:N-e,s:N-e],eta2[s:N-e,s:N-e],to_plot[s:N-e,s:N-e], cmap = "cb.iris")
CS2 = ax.contour(eta1[s:N-e,s:N-e],eta2[s:N-e,s:N-e],to_plot[s:N-e,s:N-e], levels=[1], linestyles = 'dashed')


#pnr_surf = ax.plot_surface(eta1[1:N-1-drop,1:N-1-drop],eta2[1:N-1-drop,1:N-1-drop],to_plot, label="SU(1,1)")
ax.set_xlabel('Transmisivity 1')
ax.set_ylabel('Transmisivity 2')
#ax.set_zlabel(r"$\frac{D_Q}{D}$")
cbar = fig.colorbar(CS)
cbar.ax.set_ylabel(r"$\frac{D_{\text{SU(1,1)}}}{D_{\text{Coherent}}}$")
ax.set_title("Coherent Homodyne vs Quantum Limit")

plt.show()

In [ ]:

fig = plt.figure()
ax = fig.add_subplot(projection='3d')
su11_surf = ax.plot_surface(eta1,eta2,su11_rels, label = "SU(1,1)")
coh_surf = ax.plot_surface(eta1,eta2,homo_rels, label = "Coherent")
ax.set_xlabel('Transmisivity 1')
ax.set_ylabel('Transmisivity 2')
ax.set_zlabel('Relative Entropy')
ax.set_title("SU(1,1) vs Coherent Homodyne")
ax.legend(handles = [su11_surf,coh_surf])
plt.show()